# WaferGuard: repaired Fab-7 excursion detector

This notebook repairs the supplied pre-release notebook (`bnypi.ipynb`). The pipeline code is in `wafer_guard.py` and tested in `tests/`. `AUDIT.md` has the write-up and `audit_evidence.py` reproduces the original defects.

**Task.** Predict defective lots (`label == +1`, about 6.6% of lots) from 590 sensors with a supervised autoencoder (reconstruction plus a classification head). Production causality applies: nothing from a later time t', or from the evaluation data, may influence a prediction or a preprocessing step at time t.

**Assumptions**
1. `timestamp` is the lot start time and the only ordering variable. Rows are never shuffled across splits.
2. Legacy Probe Station PS-L wafers have `sensor_0 == 0.0` exactly. On them `sensor_1` is a firmware code in {100, 200, 300}. A `NaN` in `sensor_0` is a missing value, not a legacy marker.
3. Final metrics are computed once, on the last chronological split, at its natural defect rate.
4. Everything learned (feature selection, clipping and scaling statistics, class weight, epoch, threshold) comes from training or validation rows only.

## 0. Setup

In [1]:
import inspect
import warnings
import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

import wafer_guard as wg


def show(*objs):
    """Print the source of the functions and classes the notebook relies on, straight from wafer_guard.py."""
    for o in objs:
        print(inspect.getsource(o))

warnings.filterwarnings("ignore", category=UserWarning)
CSV_PATH = "secom_telemetry_modified.csv"      # run from the repository root
SEED = wg.SEED
wg.set_seed(SEED)
DEVICE = "cpu"
print(f"torch {torch.__version__} | device={DEVICE} | seed={SEED}")
print("hyper-parameters:", wg.HP)

torch 2.14.0+cpu | device=cpu | seed=1337
hyper-parameters: {'batch_size': 64, 'epochs': 40, 'lr': 0.001, 'weight_decay': 1e-05, 'hidden': 256, 'latent': 32, 'lambda_clf': 1.0, 'dropout': 0.1}


## 1. Data inspection

In [2]:
X, y, ts = wg.load_telemetry(CSV_PATH)          # sorted by timestamp (stable)
legacy = wg.WaferPreprocessor.legacy_mask(X)
print(f"X {X.shape} | defects {y.sum()} / {len(y)} = {y.mean():.2%} | time {str(ts.min())[:16]} -> {str(ts.max())[:16]}")
print(f"missing cells {np.isnan(X).mean():.2%} | constant columns {int((np.nanstd(X, axis=0) == 0).sum())} "
      f"| columns >50% missing {int((np.isnan(X).mean(axis=0) > 0.5).sum())} | duplicate timestamps {int(pd.Series(ts).duplicated().sum())}")
print(f"legacy wafers (sensor_0 == 0.0): {legacy.sum()} | sensor_0 NaN (missing, not legacy): {int(np.isnan(X[:, 0]).sum())}")
print("legacy sensor_1 codes:", {float(c): int((X[legacy, 1] == c).sum()) for c in np.unique(X[legacy, 1])})
print(f"non-legacy sensor_1 range: {np.nanmin(X[~legacy, 1]):.1f} .. {np.nanmax(X[~legacy, 1]):.1f}  (continuous, never a code)")
print(f"defect rate  legacy {y[legacy].mean():.2%} ({y[legacy].sum()}/{legacy.sum()}) | non-legacy {y[~legacy].mean():.2%}")
month = pd.Series(ts).dt.to_period("M")
print(pd.DataFrame({"lots": pd.Series(y).groupby(month).size(), "defect_rate": pd.Series(y).groupby(month).mean().round(3),
                    "legacy_share": pd.Series(legacy).groupby(month).mean().round(3)}).T.to_string())

X (1567, 590) | defects 104 / 1567 = 6.64% | time 2008-07-19T11:55 -> 2008-10-17T06:07
missing cells 4.54% | constant columns 116 | columns >50% missing 28 | duplicate timestamps 33
legacy wafers (sensor_0 == 0.0): 146 | sensor_0 NaN (missing, not legacy): 6
legacy sensor_1 codes: {100.0: 61, 200.0: 35, 300.0: 50}
non-legacy sensor_1 range: 2158.8 .. 2846.4  (continuous, never a code)
defect rate  legacy 0.00% (0/146) | non-legacy 7.32%
              2008-07  2008-08  2008-09  2008-10
lots           63.000  555.000  590.000  359.000
defect_rate     0.222    0.092    0.029    0.061
legacy_share    0.032    0.077    0.117    0.089


The defect rate falls over time and the legacy share changes, so a random split would flatter the model. The split has to be chronological.

## 2. Chronological split (60 / 20 / 20)

In [3]:
tr, va, te = wg.chronological_split(ts, fractions=(0.6, 0.2, 0.2))
assert ts[tr].max() < ts[va].min() and ts[va].max() < ts[te].min()       # strict time order, ties never split
rows = [dict(split=n, lots=len(i), start=str(ts[i].min())[:16], end=str(ts[i].max())[:16], defects=int(y[i].sum()),
             prevalence=f"{y[i].mean():.2%}", legacy=int(legacy[i].sum())) for n, i in [("train", tr), ("val", va), ("test", te)]]
print(pd.DataFrame(rows).to_string(index=False))

split  lots            start              end  defects prevalence  legacy
train   940 2008-07-19T11:55 2008-09-20T05:34       76      8.09%      79
  val   314 2008-09-20T06:08 2008-10-02T20:54       11      3.50%      41
 test   313 2008-10-02T21:32 2008-10-17T06:07       17      5.43%      26


Train fits the preprocessor and the network. Validation picks the epoch and the decision threshold. Test is scored once, in section 7.

**Fix 1 (leakage), the split.** Source of `chronological_split`: cut points never fall inside a block of identical timestamps, and the assertion at the end enforces strict time order.

In [4]:
show(wg.chronological_split)

def chronological_split(timestamps, fractions=(0.6, 0.2, 0.2)):
    """Split time-sorted data into train / val / test index arrays.

    Cut points never fall inside a block of identical timestamps, so
    max(train time) < min(val time) < ... holds strictly.
    """
    ts = np.asarray(timestamps)
    if len(ts) > 1 and np.any(ts[1:] < ts[:-1]):
        raise ValueError("timestamps must be sorted ascending")
    if not np.isclose(sum(fractions), 1.0) or len(fractions) != 3:
        raise ValueError("fractions must be three numbers summing to 1")
    n = len(ts)
    cuts, acc = [], 0.0
    for f in fractions[:2]:
        acc += f
        c = int(round(n * acc))
        while 0 < c < n and ts[c] == ts[c - 1]:
            c += 1
        cuts.append(c)
    idx = np.arange(n)
    parts = (idx[:cuts[0]], idx[cuts[0]:cuts[1]], idx[cuts[1]:])
    if any(len(p) == 0 for p in parts):
        raise ValueError("a split is empty; adjust fractions")
    assert ts[parts[0]].max() < ts[parts[1]].min(

## 3. Preprocessing (legacy-aware, fitted on train only)

In [5]:
prep = wg.WaferPreprocessor(max_missing=0.5).fit(X[tr])                  # training rows only
Z_tr, Z_va, Z_te = (prep.transform(X[i]) for i in (tr, va, te))          # frozen transform reused everywhere
print(f"features: {Z_tr.shape[1]} = {len(prep.keep_)} continuous + is_legacy + 3 firmware-code one-hots + code_unknown")
print(f"dropped (constant or >50% missing in TRAIN): {X.shape[1] - len(prep.keep_)} of {X.shape[1]} sensors")
k0, k1 = list(prep.keep_).index(0), list(prep.keep_).index(1)
print("legacy rows -> is_legacy:", int(Z_tr[legacy[tr], -5].sum()), "of", int(legacy[tr].sum()),
      "| code one-hots (100/200/300/unknown):", Z_tr[legacy[tr], -4:].sum(axis=0).astype(int).tolist())
print("continuous sensor_0 / sensor_1 channels on legacy rows are masked to 0.0:",
      bool(np.all(Z_tr[legacy[tr]][:, [k0, k1]] == 0.0)))
print(f"sensor_0 / sensor_1 scaling stats come from non-legacy rows: mean {prep.mean_[0]:.1f} / {prep.mean_[1]:.1f}")
assert np.isfinite(Z_tr).all() and np.isfinite(Z_va).all() and np.isfinite(Z_te).all()

features: 449 = 444 continuous + is_legacy + 3 firmware-code one-hots + code_unknown
dropped (constant or >50% missing in TRAIN): 146 of 590 sensors
legacy rows -> is_legacy: 79 of 79 | code one-hots (100/200/300/unknown): [33, 20, 26, 0]
continuous sensor_0 / sensor_1 channels on legacy rows are masked to 0.0: True
sensor_0 / sensor_1 scaling stats come from non-legacy rows: mean 3009.5 / 2495.1


**Fix 1 (leakage) and Fix 3 (legacy wafers), the preprocessor.** `fit` sees training rows only and learns the clipping bounds, means, standard deviations and the kept features. `transform` is row-wise and frozen, so it cannot look at later rows. `legacy_mask` is the sentinel rule (`sensor_0 == 0.0`, with NaN treated as missing). On legacy rows `sensor_1` becomes a one-hot firmware code and both continuous channels are masked.

In [6]:
show(wg.WaferPreprocessor)

class WaferPreprocessor:
    """Legacy-aware, training-only-fitted feature pipeline.

    Output columns: [continuous sensors kept | is_legacy | code_100 | code_200 | code_300 | code_unknown].

    * Legacy wafers (sensor_0 == 0.0) are flagged; sensor_0 / sensor_1 on those rows
      are NOT continuous measurements. sensor_1 is one-hot encoded as a firmware code
      (unknown / missing code -> `code_unknown`), and the two continuous channels are
      masked (set to 0.0 in the standardised space) on those rows.
    * Statistics of sensor_0 / sensor_1 are fitted on non-legacy rows only.
    * Other sensors: inf -> NaN, winsorise to training [q_lo, q_hi], standardise with
      training mean/std, missing -> 0.0 (training mean). Features that are constant or
      missing in more than `max_missing` of training rows are dropped.
    """

    def __init__(self, max_missing: float = 0.5, clip_quantiles=(0.01, 0.99),
                 firmware_codes=FIRMWARE_CODES):
        self.max_missing =

## 4. Model: supervised autoencoder

Same architecture as the original: a `Linear-LayerNorm-GELU` encoder down to a 32-dimensional latent vector, a mirrored decoder, and a linear classification head. The one structural change is that tensors are strictly `[B, F]`. The original's extra channel axis (`[B, 1, F]`) caused the silent broadcast in the loss. The model and the loss now raise on any other shape.

| tensor | shape |
|---|---|
| input `x`, reconstruction | `[B, F]` |
| latent `z` | `[B, 32]` |
| logits, target `y` | `[B]` |

In [7]:
model = wg.WaferSAE(Z_tr.shape[1], wg.HP["hidden"], wg.HP["latent"], wg.HP["dropout"])
xb = torch.from_numpy(Z_tr[:8])
recon, logits, z = model(xb)
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"shapes  x {tuple(xb.shape)} | recon {tuple(recon.shape)} | z {tuple(z.shape)} | logits {tuple(logits.shape)}")

parameters: 305,666
shapes  x (8, 449) | recon (8, 449) | z (8, 32) | logits (8,)


Source of the model. `forward` raises unless the input is exactly `[B, F]`, which is what stops the silent broadcast.

In [8]:
show(wg.WaferSAE)

class WaferSAE(nn.Module):
    """Supervised autoencoder (architecture of the original notebook, with shape guards)."""

    def __init__(self, d_in: int, d_hid: int = 256, d_lat: int = 32, dropout: float = 0.10):
        super().__init__()
        self.d_in = d_in
        self.encoder = nn.Sequential(
            nn.Linear(d_in, d_hid), nn.LayerNorm(d_hid), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_hid, d_hid // 2), nn.LayerNorm(d_hid // 2), nn.GELU(),
            nn.Linear(d_hid // 2, d_lat))
        self.decoder = nn.Sequential(
            nn.Linear(d_lat, d_hid // 2), nn.GELU(),
            nn.Linear(d_hid // 2, d_hid), nn.GELU(),
            nn.Linear(d_hid, d_in))
        self.head = nn.Linear(d_lat, 1)

    def forward(self, x):
        if x.ndim != 2 or x.shape[1] != self.d_in:
            raise ValueError(f"expected x of shape [B, {self.d_in}], got {tuple(x.shape)}")
        z = self.encoder(x)                                   # [B, d_lat]
        recon = self.

## 5. Training and loss

- Loss is per-sample reconstruction MSE plus `BCEWithLogits(logits [B], y [B])`.
- Class imbalance is handled with `pos_weight = n_neg / n_pos`, computed on the training split. SMOTE is not used (see `AUDIT.md`).
- The first step asserts that every trainable parameter received a finite gradient. A non-finite loss raises.
- The epoch with the best validation PR-AUC is kept.

**Fix 2 (loss), the source.** `sae_loss` checks every shape before computing anything, so a `[B, 1, F]` target raises instead of broadcasting to `[B, B, F]`. Reconstruction error is averaged per sample, then over the batch.

In [9]:
show(wg.sae_loss, wg.assert_valid_gradients, wg.train_model)

def sae_loss(recon, logits, x, y, pos_weight=None, lambda_clf: float = 1.0):
    """Reconstruction MSE + weighted BCE with strict shape checks (no silent broadcasting)."""
    if x.ndim != 2:
        raise ValueError(f"x must be [B, F]; got {tuple(x.shape)} (a channel axis would broadcast)")
    if recon.shape != x.shape:
        raise ValueError(f"recon {tuple(recon.shape)} != x {tuple(x.shape)}")
    if logits.shape != (x.shape[0],) or y.shape != logits.shape:
        raise ValueError(f"logits {tuple(logits.shape)} / y {tuple(y.shape)} must both be [{x.shape[0]}]")
    recon_loss = (recon - x).pow(2).mean(dim=1).mean()        # per-sample MSE, then batch mean
    clf_loss = F.binary_cross_entropy_with_logits(logits, y, pos_weight=pos_weight)
    total = recon_loss + lambda_clf * clf_loss
    if not torch.isfinite(total):
        raise FloatingPointError(f"non-finite loss: recon={recon_loss.item()}, clf={clf_loss.item()}")
    return total, recon_loss, clf_loss

def assert_valid_gradi

In [10]:
model, history = wg.train_model(Z_tr, y[tr], Z_va, y[va], wg.HP, seed=SEED, device=DEVICE, verbose=True)
best = max(history, key=lambda h: h["val_ap"])
print(f"best epoch by validation PR-AUC: {best['epoch']} (val AP {best['val_ap']:.4f}; val prevalence {y[va].mean():.4f})")

epoch 01/40 | loss 2.2103 | recon 0.9678 | clf 1.2425 | val AP 0.0530


epoch 10/40 | loss 0.7274 | recon 0.7248 | clf 0.0026 | val AP 0.0408


epoch 20/40 | loss 0.6090 | recon 0.6075 | clf 0.0015 | val AP 0.0369


epoch 30/40 | loss 0.5661 | recon 0.5651 | clf 0.0011 | val AP 0.0363


epoch 40/40 | loss 0.5570 | recon 0.5562 | clf 0.0008 | val AP 0.0361
best epoch by validation PR-AUC: 1 (val AP 0.0530; val prevalence 0.0350)


## 6. Validation and threshold

In [11]:
show(wg.select_threshold)

def select_threshold(y_val, p_val) -> float:
    """F1-maximising threshold chosen on validation data only (ties -> largest threshold)."""
    y_val = np.asarray(y_val)
    if y_val.sum() == 0:
        raise ValueError("validation split has no positives; cannot select a threshold")
    prec, rec, thr = precision_recall_curve(y_val, p_val)
    f1 = 2 * prec[:-1] * rec[:-1] / np.maximum(prec[:-1] + rec[:-1], 1e-12)
    return float(thr[np.flatnonzero(f1 == f1.max())[-1]])



In [12]:
p_va = wg.predict_proba(model, Z_va, device=DEVICE)
threshold = wg.select_threshold(y[va], p_va)       # F1-maximising threshold on validation only
val = wg.evaluate(y[va], p_va, threshold)
print(f"threshold {threshold:.4f} | val: precision {val['precision']:.3f} recall {val['recall']:.3f} F1 {val['f1']:.3f} "
      f"PR-AUC {val['pr_auc']:.3f} (prevalence {val['prevalence']:.3f}) | TP {val['tp']} FP {val['fp']} FN {val['fn']} TN {val['tn']}")

threshold 0.6162 | val: precision 0.077 recall 0.182 F1 0.108 PR-AUC 0.053 (prevalence 0.035) | TP 2 FP 24 FN 9 TN 279


## 7. Final evaluation on the untouched test split

In [13]:
p_te = wg.predict_proba(model, Z_te, device=DEVICE)
test = wg.evaluate(y[te], p_te, threshold)
assert test["n"] == len(te) and abs(test["prevalence"] - y[te].mean()) < 1e-12     # test set is not resampled
ci = wg.bootstrap_ci(y[te], p_te, threshold, n_boot=2000, seed=SEED)
print(f"test lots {test['n']} | defects {test['n_pos']} | passes {test['n_neg']} | natural prevalence {test['prevalence']:.2%}"
      f"  (whole dataset {y.mean():.2%})")
print(f"precision {test['precision']:.3f} | recall {test['recall']:.3f} | F1 {test['f1']:.3f} | PR-AUC {test['pr_auc']:.3f}  "
      f"[no-skill PR-AUC = prevalence = {test['prevalence']:.3f}]")
print(f"supplementary: ROC-AUC {test['roc_auc']:.3f} | accuracy {test['accuracy']:.3f} (an all-pass classifier scores {1 - test['prevalence']:.3f})")
print(f"confusion matrix  [[TN {test['tn']}, FP {test['fp']}], [FN {test['fn']}, TP {test['tp']}]]")
print(f"bootstrap 95% CI ({ci['n_valid']} valid draws): F1 {ci['f1'][0]:.3f}..{ci['f1'][1]:.3f} | PR-AUC {ci['pr_auc'][0]:.3f}..{ci['pr_auc'][1]:.3f}")

test lots 313 | defects 17 | passes 296 | natural prevalence 5.43%  (whole dataset 6.64%)
precision 0.095 | recall 0.118 | F1 0.105 | PR-AUC 0.081  [no-skill PR-AUC = prevalence = 0.054]
supplementary: ROC-AUC 0.422 | accuracy 0.891 (an all-pass classifier scores 0.946)
confusion matrix  [[TN 277, FP 19], [FN 15, TP 2]]
bootstrap 95% CI (2000 valid draws): F1 0.000..0.255 | PR-AUC 0.032..0.236


A false negative is a defective lot that ships. A false positive sends a good lot to extra inspection. At about 5% prevalence even a small false-alarm rate produces more false alarms than catches, so precision is the number to watch, not accuracy. The threshold maximises F1 on validation.

### Baselines (same features, splits and threshold rule)

In [14]:
prevalence_f1 = 2 * y[te].mean() / (1 + y[te].mean())            # flag every lot as defective
print(f"flag-everything baseline: precision {y[te].mean():.3f} recall 1.000 F1 {prevalence_f1:.3f} | no-skill PR-AUC {y[te].mean():.3f}")

# class-weighted logistic regression; C is chosen on validation PR-AUC only
best_lr = max((average_precision_score(y[va], LogisticRegression(C=C, class_weight="balanced", max_iter=3000)
                                       .fit(Z_tr, y[tr]).predict_proba(Z_va)[:, 1]), C) for C in (1e-4, 1e-3, 1e-2, 1e-1, 1.0))
lr = LogisticRegression(C=best_lr[1], class_weight="balanced", max_iter=3000).fit(Z_tr, y[tr])
thr_lr = wg.select_threshold(y[va], lr.predict_proba(Z_va)[:, 1])
m = wg.evaluate(y[te], lr.predict_proba(Z_te)[:, 1], thr_lr)
print(f"logistic regression (C={best_lr[1]:g}): precision {m['precision']:.3f} recall {m['recall']:.3f} F1 {m['f1']:.3f} "
      f"PR-AUC {m['pr_auc']:.3f} ROC-AUC {m['roc_auc']:.3f} | TP {m['tp']} FP {m['fp']} FN {m['fn']}")

flag-everything baseline: precision 0.054 recall 1.000 F1 0.103 | no-skill PR-AUC 0.054
logistic regression (C=0.0001): precision 0.076 recall 0.294 F1 0.120 PR-AUC 0.073 ROC-AUC 0.579 | TP 5 FP 61 FN 12


### Seed sensitivity (full pipeline re-run per seed)

In [15]:
rows = []
for s in (SEED, 1, 2, 3, 4):
    r = wg.run_experiment(CSV_PATH, seed=s)
    t = r["test"]
    rows.append(dict(seed=s, threshold=round(r["threshold"], 3), precision=round(t["precision"], 3), recall=round(t["recall"], 3),
                     f1=round(t["f1"], 3), pr_auc=round(t["pr_auc"], 3), roc_auc=round(t["roc_auc"], 3)))
    if s == SEED:                                                  # the helper must reproduce the step-by-step run above
        assert abs(t["f1"] - test["f1"]) < 1e-12 and abs(t["pr_auc"] - test["pr_auc"]) < 1e-12
sweep = pd.DataFrame(rows)
print(sweep.to_string(index=False))
print(sweep[["precision", "recall", "f1", "pr_auc", "roc_auc"]].agg(["mean", "std"]).round(3).to_string())

 seed  threshold  precision  recall    f1  pr_auc  roc_auc
 1337      0.616      0.095   0.118 0.105   0.081    0.422
    1      0.584      0.057   0.176 0.086   0.068    0.521
    2      0.005      0.115   0.176 0.140   0.075    0.530
    3      0.433      0.067   0.529 0.118   0.079    0.581
    4      0.018      0.059   0.471 0.105   0.088    0.593
      precision  recall     f1  pr_auc  roc_auc
mean      0.079   0.294  0.111   0.078    0.529
std       0.025   0.191  0.020   0.007    0.068


## 8. Artifacts and reload check

In [16]:
wg.save_artifacts("artifacts", model, prep, threshold)
model2, prep2, thr2 = wg.load_artifacts("artifacts")
p_te2 = wg.predict_proba(model2, prep2.transform(X[te]))
assert thr2 == threshold and np.array_equal(p_te2, p_te)
print("saved to ./artifacts and reloaded: identical test probabilities, same threshold")

saved to ./artifacts and reloaded: identical test probabilities, same threshold


## 9. Rolling out-of-time backtest

One test split holds 17 defects, so a single F1 can be luck. This section repeats the evaluation on five consecutive future windows of 188 lots each. In every window the preprocessor and each model are fitted on earlier lots only, and rank metrics (ROC-AUC, PR-AUC) are reported because they need no threshold. Logistic regression, a random forest and gradient boosting are included as references; the repaired autoencoder is the same model as above. This cell takes two to three minutes.

In [17]:
import backtest as bt
bres = bt.run_backtest(CSV_PATH)
bt.print_table(bres)

windows: [188, 188, 188, 188, 188] lots, defects [4, 7, 6, 18, 4] (total 39), mean prevalence 0.041
model                  mean ROC-AUC           95% CI  mean PR-AUC   per-window ROC-AUC
repaired_sae                  0.566     0.46 to 0.67        0.075   0.49 0.56 0.72 0.57 0.48
logistic_regression           0.638     0.56 to 0.72        0.080   0.54 0.67 0.69 0.58 0.71
random_forest                 0.626     0.54 to 0.71        0.149   0.50 0.96 0.59 0.56 0.52
gradient_boosting             0.594     0.51 to 0.68        0.171   0.39 0.97 0.60 0.59 0.41


Reading the table: a ROC-AUC of 0.5 is chance, and an interval that includes 0.5 means the model has not shown skill. Each window holds only 4 to 18 defects, so the per-window values swing widely and the PR-AUC averages are driven by one or two windows.

## 10. Result

In [18]:
print("FINAL (seed %d, chronological test split, natural prevalence)" % SEED)
print(f"  lots {test['n']} | defects {test['n_pos']} ({test['prevalence']:.2%}) | threshold {threshold:.4f}")
print(f"  precision {test['precision']:.3f} | recall {test['recall']:.3f} | F1 {test['f1']:.3f} | PR-AUC {test['pr_auc']:.3f} | ROC-AUC {test['roc_auc']:.3f}")
print(f"  TN {test['tn']} FP {test['fp']} FN {test['fn']} TP {test['tp']}")
print(f"  original release gate F1 >= 0.90: {'MET' if test['f1'] >= 0.90 else 'NOT MET'}")

FINAL (seed 1337, chronological test split, natural prevalence)
  lots 313 | defects 17 (5.43%) | threshold 0.6162
  precision 0.095 | recall 0.118 | F1 0.105 | PR-AUC 0.081 | ROC-AUC 0.422
  TN 277 FP 19 FN 15 TP 2
  original release gate F1 >= 0.90: NOT MET


**Reading the result**
- The test split has only a handful of defective lots, so every test metric is fragile (see the bootstrap interval and the seed table).
- On this split the model is not distinguishable from no-skill: test F1 is about that of flagging every lot, the PR-AUC interval includes values near the prevalence, ROC-AUC is near 0.5 across seeds, and validation PR-AUC peaks in the first epoch while the network memorises the training defects.
- Across the five windows of section 9, the repaired autoencoder is the weakest of the four models and its ROC-AUC interval includes 0.5. The simpler models show a weak but real out-of-time signal, with intervals above 0.5 but modest precision. None of them is close to ship-ready, and the signal is unstable from window to window.
- The original 0.90+ came from leakage, a 50%-prevalence synthetic-heavy test set and a corrupted loss. It cannot be recovered honestly.
- All 146 legacy wafers pass electrical test. `is_legacy` is informative, but there are no PS-L defects to learn failure behaviour from.
- The threshold was tuned on a validation split with 11 defects, so it will not hold up under drift. A deployment needs recalibration and monitoring of the defect rate.
- Hyperparameters are the original notebook's. Nothing was tuned against the test split.